In [ ]:
import pandas as pd
import warnings
from model import NN
from utils import LoadData
from rolling_train_test import RollingTrainTest
import os
warnings.simplefilter(action='ignore', category=FutureWarning)

In [ ]:
data_list = [
    'factor_cleaned_structure_R.csv',
    'factor_cleaned_cash_R.csv',
    'factor_cleaned_debtpay_R.csv',
    'factor_cleaned_divident_R.csv',
    'factor_cleaned_growth_R.csv',
    'factor_cleaned_profit_quality_R.csv',
    'factor_cleaned_profit_R.csv',
    'factor_cleaned_relative_R.csv',
    'factor_cleaned_PS_R.csv',
    'factor_cleaned_operatin_R.csv'
]

for _ in data_list:
    factor = pd.read_csv(f"../CSV/{_}")
    input_dim = factor.shape[1] - 2
    # print(f"Input dimension: {input_dim}")

    label = pd.read_csv('../CSV/label_cleaned.csv')
    # label.describe()
    target = 1

    # load data and model
    Data = LoadData(factor, label, batch_size=32, num_workers=0, shuffle=True)
    model_list = [
        NN(input_dim, target=target, alpha=0.8, l1_ratio=0.5, layer=5, model_name="NN")
    ]

    # rolling test
    count = 0
    profit_list = []
    for model in model_list:
        RTT = RollingTrainTest(model, Data, train_size=0.5, test_size=0.1, epochs=10, patience=3, criterion=None, count=count)
        RTT.info(
            predictability_name = f"[--importance test--|--{_}--]"
            )
        RTT.run()
        RTT.backtest(trade_mode=2)
        count += 1
        print(f"Model {model.model_name} backtest completed...")
        profit_list.append(RTT.SR)

    # 2\3\4层神经网络的平均值
    standard_SR = round((4.4967+4.5013+4.4978)/3, 4)
    
    test_SR = round(sum(profit_list)/len(profit_list), 4)
    importance = standard_SR - test_SR
    print (f"{_}: {importance}")
    print("=" * 50)

    # write into CSV
    file_path = '../CSV/importance.csv'
    mode = 'a' if os.path.exists(file_path) else 'w'      
    with open(file_path, mode) as f:
        if mode == 'w':
            f.write('predictability_name,test_SR,importance\n')
        f.write(f'{_},{test_SR:.4f},{importance:.4f}\n')
